# 在 Google Colab 上运行 Jasna

In [ ]:
# 1. 检查 GPU 信息
!nvidia-smi

Wed Mar  4 15:16:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 安装 mkvmerge

In [ ]:
# 2. 安装 mkvmerge
!apt-get update -qq
!apt-get install -y -qq mkvtoolnix

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../0-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libdvdread8:amd64.
Preparing to unpack .../1-libdvdread8_6.1.2-1_amd64.deb ...
Unpacking libdvdread8:amd64 (6.1.2-1) ...
Selecting previously unselected package libebml5:amd64.
Preparing to unpack .../2-libebml5_1.4.2-2_amd64.deb ...
Unpacking libebml5:amd64 (1.4.2-2) ...
Selecting previously unselected package libfmt8:amd64.
Preparing to unpack .../3-libfmt8_8.1.1+ds1-2_amd64.deb ...
Unpacking libfmt8:amd64 (8.1.1+ds1-2) ...
Selecting previously unselected package libmatroska7:amd64.
Preparing to unpa

## 安装 FFmpeg 和 FFprobe

In [ ]:
# 3. 安装 FFmpeg
!apt-get install -y -qq build-essential yasm cmake libtool libc6 libc6-dev unzip wget libnuma1 libnuma-dev git

# 使用 BtbN 提供的支持 NVENC/NVDEC 的静态构建
!wget -q --show-progress https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz
!tar -xf ffmpeg-master-latest-linux64-gpl.tar.xz
!cp -f ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg
!cp -f ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe
!chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe
!rm -rf ffmpeg-master-latest-linux64-gpl*

# 验证 FFmpeg 版本及 NVENC 支持
!ffmpeg -version | head -3
!echo "--- NVENC 编码器 ---"
!ffmpeg -encoders 2>/dev/null | grep nvenc

Preconfiguring packages ...
(Reading database ... 122393 files and directories currently installed.)
Preparing to unpack .../libc6-dev_2.35-0ubuntu3.13_amd64.deb ...
Unpacking libc6-dev:amd64 (2.35-0ubuntu3.13) over (2.35-0ubuntu3.9) ...
Preparing to unpack .../libc-dev-bin_2.35-0ubuntu3.13_amd64.deb ...
Unpacking libc-dev-bin (2.35-0ubuntu3.13) over (2.35-0ubuntu3.9) ...
Preparing to unpack .../libc6_2.35-0ubuntu3.13_amd64.deb ...
Unpacking libc6:amd64 (2.35-0ubuntu3.13) over (2.35-0ubuntu3.9) ...
Setting up libc6:amd64 (2.35-0ubuntu3.13) ...
(Reading database ... 122393 files and directories currently installed.)
Preparing to unpack .../git_1%3a2.34.1-1ubuntu1.17_amd64.deb ...
Unpacking git (1:2.34.1-1ubuntu1.17) over (1:2.34.1-1ubuntu1.15) ...
Selecting previously unselected package libtool.
Preparing to unpack .../libtool_2.4.6-15build2_all.deb ...
Unpacking libtool (2.4.6-15build2) ...
Selecting previously unselected package yasm.
Preparing to unpack .../yasm_1.3.0-2.1_amd64.deb .

## 从源码编译 Jasna

In [ ]:
# 4. 下载并部署 Jasna
# 准备全局构建工具
!pip install -q uv cmake ninja

# 拉取 Jasna 源码
%cd /content
!rm -rf jasna vali PyNvVideoCodec # 清理之前的错误尝试
!git clone https://github.com/Kruk2/jasna.git
%cd jasna

# 建立虚拟环境
!uv python install 3.13
!uv venv ./.venv --python 3.13

# 设置前置依赖
!uv pip install --python /content/jasna/.venv "setuptools<70.0.0"
!uv pip install --python /content/jasna/.venv wheel numpy torch torchvision torchaudio scikit-build

# 克隆两个底层 Nvidia 库
%cd /content
!git clone https://codeberg.org/Kruk2/vali.git
!git clone https://codeberg.org/Kruk2/PyNvVideoCodec.git

# 安装 FFmpeg 的 C/C++ 开发头文件
!apt-get update -qq
!apt-get install -y -qq pkg-config libavformat-dev libavcodec-dev libavutil-dev libswscale-dev libswresample-dev

# 获取 FFmpeg 8+ 的动态开发包（包含头文件和 .so 库）
%cd /content
!wget -qO ffmpeg-shared.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl-shared.tar.xz
!tar -xf ffmpeg-shared.tar.xz
!apt-get update -qq
!apt-get install -y -qq pkg-config

# 将最新的头文件和动态库注入系统核心目录
# 使用 -a 参数确保软链接不被破坏
!cp -a ffmpeg-master-latest-linux64-gpl-shared/include/* /usr/local/include/
!cp -a ffmpeg-master-latest-linux64-gpl-shared/lib/* /usr/local/lib/
!ldconfig  # 刷新系统的动态库缓存

# 进入 vali 目录，关闭隔离并指定使用 Jasna 的虚拟环境进行编译
%cd /content/vali

# 拉取缺失的外部子模块 (比如 dlpack)
!git submodule update --init --recursive

# 强制引导编译器去刚才注入的目录寻找最新版 FFmpeg
!PKG_CONFIG_PATH="/usr/local/lib/pkgconfig" uv pip install --python /content/jasna/.venv . --no-build-isolation

# 进入 PyNvVideoCodec 目录编译
%cd /content/PyNvVideoCodec
!uv pip install --python /content/jasna/.venv . --no-build-isolation

# 回到 Jasna 完成最终本体安装
%cd /content/jasna
!uv pip install --python /content/jasna/.venv -e .[dev]
!ls -s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.8 MB/s eta 0:00:00
/content
Cloning into 'jasna'...
remote: Enumerating objects: 1404, done.
remote: Counting objects: 100% (690/690), done.
remote: Compressing objects: 100% (303/303), done.
remote: Total 1404 (delta 494), reused 548 (delta 384), pack-reused 714 (from 1)
Receiving objects: 100% (1404/1404), 4.99 MiB | 20.35 MiB/s, done.
Resolving deltas: 100% (919/919), done.
/content/jasna
Installed Python 3.13.12 in 1.08s
 + cpython-3.13.12-linux-x86_64-gnu (python3.13)
Using CPython 3.13.12
Creating virtual environment at: ./.venv
Activate with: source .venv/bin/activate
Resolved 1 package in 40ms
Prepared 1 package in 35ms
Installed 1 package in 7ms
 + setuptools==69.5.1
Resolved 36 packages in 116ms
Prepared 35 packages in 33.52s
Installed 35 packages in 235ms
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.4.0
 + distro==1.9.0
 + filelock==

In [ ]:
# 验证安装
!./.venv/bin/python -m jasna --help

usage: jasna [-h] [--version] [--benchmark] [--input INPUT] [--output OUTPUT]
             [--batch-size BATCH_SIZE] [--device DEVICE] [--fp16 | --no-fp16]
             [--log-level {debug,info,warning,error}] [--disable-ffmpeg-check]
             [--restoration-model-name {basicvsrpp}]
             [--restoration-model-path RESTORATION_MODEL_PATH]
             [--compile-basicvsrpp | --no-compile-basicvsrpp]
             [--max-clip-size MAX_CLIP_SIZE]
             [--temporal-overlap TEMPORAL_OVERLAP]
             [--enable-crossfade | --no-enable-crossfade]
             [--denoise {none,low,medium,high}]
             [--denoise-step {after_primary,after_secondary}]
             [--secondary-restoration {none,swin2sr,tvai}]
             [--swin2sr-batch-size SWIN2SR_BATCH_SIZE]
             [--swin2sr-compilation | --no-swin2sr-compilation]
             [--tvai-ffmpeg-path TVAI_FFMPEG_PATH] [--tvai-model TVAI_MODEL]
             [--tvai-scale {1,2,4}] [--tvai-args TVAI_ARGS]
        

In [ ]:
import os
import glob

# 指向 Python 3.13 虚拟环境中的依赖包目录
site_packages = "/content/jasna/.venv/lib/python3.13/site-packages"

# 自动抓取所有通过 pip 安装的底层 Nvidia 库和 TensorRT 库路径
nvidia_libs = glob.glob(f"{site_packages}/nvidia/*/lib")
tensorrt_libs = glob.glob(f"{site_packages}/tensorrt*")

# 组合成新的 LD_LIBRARY_PATH
# 重点：把虚拟环境的库排在最前面，末尾保留底层的驱动路径 /usr/lib64-nvidia
# 绝不能加入 /usr/local/cuda，避免与系统自带的 CUDA 12 冲突
custom_ld_path = ":".join(nvidia_libs + tensorrt_libs) + ":/usr/lib64-nvidia"

# 写入当前环境
os.environ['LD_LIBRARY_PATH'] = custom_ld_path
print("✅ 环境变量已更新！系统现在将使用虚拟环境中的 CUDA 13 库。")

# 验证修复结果：测试虚拟环境中的 PyTorch 是否能正常识别 L4 显卡
print("\n测试 PyTorch GPU 识别情况：")
!/content/jasna/.venv/bin/python -c "import torch; print('🚀 显卡可用:', torch.cuda.is_available()); print('🖥️ 显卡型号:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无')"

✅ 环境变量已更新！系统现在将使用虚拟环境中的 CUDA 13 库。

测试 PyTorch GPU 识别情况：
🚀 显卡可用: True
🖥️ 显卡型号: NVIDIA L4


## 挂载 Cloud Storage Bucket 以存储权重和引擎

In [ ]:
# 5. 挂载 Cloud Storage Bucket
# 身份验证
from google.colab import auth
auth.authenticate_user()

# 配置仓库并安装 gcsfuse
!echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!apt -qq update && apt -qq install gcsfuse

# 在 Colab 磁盘上创建挂载点
!mkdir -p /content/gcs_bucket

# 挂载 Bucket（不要带 gs:// 前缀）
!gcsfuse --implicit-dirs YOUR_BUCKET_NAME /content/gcs_bucket

deb https://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100  1022  100  1022    0     0  13625      0 --:--:-- --:--:-- --:--:-- 13810
OK
93 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following packages were automatically installed and are no longer r

In [ ]:
# 为权重文件夹构建软连接
# 移动文件
!mkdir -p /content/gcs_bucket/model_weights

# 创建软连接
!rm -rf ./model_weights
!ln -s /content/gcs_bucket/model_weights ./model_weights
!ls -l

total 80
drwxr-xr-x  2 root root  4096 Mar  4 15:17 assets
-rw-r--r--  1 root root  2022 Mar  4 15:17 build_exe.py
-rw-r--r--  1 root root  1785 Mar  4 15:17 evaluate_rfdetr.py
drwxr-xr-x 12 root root  4096 Mar  4 15:21 jasna
drwxr-xr-x  2 root root  4096 Mar  4 15:20 jasna.egg-info
-rw-r--r--  1 root root  3225 Mar  4 15:17 jasna.spec
-rw-r--r--  1 root root 34523 Mar  4 15:17 LICENSE
lrwxrwxrwx  1 root root    33 Mar  4 15:24 model_weights -> /content/gcs_bucket/model_weights
drwxr-xr-x  2 root root  4096 Mar  4 15:17 patches
-rw-r--r--  1 root root   890 Mar  4 15:17 pyproject.toml
-rw-r--r--  1 root root  5918 Mar  4 15:17 README.md
drwxr-xr-x  2 root root  4096 Mar  4 15:17 tests


## 下载权重

In [ ]:
# 6. 下载修复模型和检测的权重
# 检查核心修复模型是否存在，不存在则下载
!if [ ! -f "/content/jasna/model_weights/lada_mosaic_restoration_model_generic_v1.2.pth" ]; then \
    echo "⚠️ 未发现修复模型权重，正在从云端下载 (约 78MB)..."; \
    wget -q --show-progress -O /content/jasna/model_weights/lada_mosaic_restoration_model_generic_v1.2.pth "https://huggingface.co/ladaapp/lada/resolve/main/lada_mosaic_restoration_model_generic_v1.2.pth?download=true"; \
else \
    echo "✅ 修复模型权重 (.pth) 已存在于 GCS。"; \
fi

# 检查检测模型（0.5.0-alpha3 默认使用最新的 v4 版本）
!if [ ! -f "/content/jasna/model_weights/rfdetr-v4.onnx" ]; then \
    echo "⚠️ 未发现最新的 rfdetr-v4.onnx 检测模型，正在从云端下载 ..."; \
    wget -q --show-progress -O /content/jasna/model_weights/rfdetr-v4.onnx "https://github.com/Kruk2/jasna/releases/download/v0.5.0-alpha3/rfdetr-v4.onnx"; \
else \
    echo "✅ rfdetr-v4 检测模型已存在于 GCS。"; \
fi

# 打印最终的目录状态，确认软链接和文件都正确
!echo -e "\n当前 model_weights 目录状态："
!ls -lh ./model_weights
!ls ./model_weights/ -l

✅ 修复模型权重 (.pth) 已存在于 GCS。
✅ rfdetr-v4 检测模型已存在于 GCS。

当前 model_weights 目录状态：
lrwxrwxrwx 1 root root 33 Mar  4 15:24 ./model_weights -> /content/gcs_bucket/model_weights
total 3534137
-rw-r--r-- 1 root root  263419507 Mar  1 09:51 lada_mosaic_restoration_model_generic_v1.2_clip10.trt_fp16.linux.engine
-rw-r--r-- 1 root root 3050576834 Mar  1 11:18 lada_mosaic_restoration_model_generic_v1.2_clip120.trt_fp16.linux.engine
-rw-r--r-- 1 root root   78441770 Feb 28 10:19 lada_mosaic_restoration_model_generic_v1.2.pth
-rw-r--r-- 1 root root   77893468 Mar  1 11:32 rfdetr-v4.bs4.fp16.linux.engine
-rw-r--r-- 1 root root  148623615 Feb 28 10:07 rfdetr-v4.onnx


## 使用说明
上传视频到 Colab 后，运行以下命令进行修复：
```python
%cd /content/jasna

!./.venv/bin/python -m jasna \
  --input "/content/input_video.mp4" \
  --output "/content/output_video.mp4" \
  --max-clip-size 180 \
  --detection-model rfdetr-v4 \
```

也可以挂载 Google Drive：
```python
from google.colab import drive
drive.mount('/content/drive')
```
然后指定 Drive 中的视频路径即可。

## 批量处理 Drive 中多个文件夹的视频
将下方 `FOLDERS` 列表修改为你要处理的文件夹路径，脚本会自动遍历每个文件夹中的所有视频文件并依次去马赛克。
输出文件保存在原视频所在目录，文件名格式为 `原文件名.MR.mp4`。

In [ ]:
import os
from pathlib import Path

JASNA_DIR = "/content/jasna"
JASNA_PYTHON = "/content/jasna/.venv/bin/python"
LOCAL_WORK_DIR = Path("/content/video_work")  # Colab 本地磁盘工作目录

# ====== 修改这里：添加你要处理的 Drive 文件夹路径 ======
FOLDERS = [
    "/content/drive/MyDrive/路径1/文件夹1",
    "/content/drive/MyDrive/路径1/文件夹1",
]

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".wmv", ".flv", ".mov", ".ts", ".m4v", ".webm", ".rmvb"}

# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 预览各文件夹的待处理情况
for folder in FOLDERS:
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"⚠️ 文件夹不存在: {folder}")
        continue
    videos = [f for f in sorted(folder_path.iterdir())
              if f.suffix.lower() in VIDEO_EXTENSIONS and not f.stem.endswith(".MR")]
    already = [f for f in sorted(folder_path.iterdir())
               if f.suffix.lower() in VIDEO_EXTENSIONS and f.stem.endswith(".MR")]
    skippable = sum(1 for v in videos if (v.parent / f"{v.stem}.MR.mp4").exists())
    print(f"\n📁 {folder}")
    print(f"   视频总数: {len(videos)}  已有输出(跳过): {skippable}  待处理: {len(videos) - skippable}")

In [ ]:
import os, shutil, subprocess, time, re
from pathlib import Path

JASNA_DIR = "/content/jasna"
JASNA_PYTHON = "/content/jasna/.venv/bin/python"
LOCAL_WORK_DIR = Path("/content/video_work")
os.chdir(JASNA_DIR)

# 匹配格式：
# Processing video:   0%|          |Processed: 00:01 (60f) | Remaining: 2:34:49 (373930f) | Speed: 40.3fps
# Processing video: 100%|██████████|Processed: 3:27:59 (521879f) | Remaining: 0:00 (3f) | Speed: 60.7fps
PROGRESS_RE = re.compile(
    r"Processing video:\s*(\d+)%\|[^|]*\|Processed:\s*([\d:]+)\s*\(\d+f\)\s*\|\s*Remaining:\s*([\d:?][^\|]*?)\s*\|\s*Speed:\s*(\S+)"
)

def process_video(local_input, local_output):
    """调用 Jasna 处理单个视频，实时解析并打印进度。"""
    proc = subprocess.Popen(
        [
            JASNA_PYTHON, "-m", "jasna",
            "--input", str(local_input),
            "--output", str(local_output),
            "--max-clip-size", "180",
            "--temporal-overlap", "15",
            "--encoder-settings", "cq=28",
            "--detection-model", "rfdetr-v4",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    last_pct = -1

    for raw_line in proc.stdout:
        segments = raw_line.replace('\r', '\n').splitlines()
        for seg in segments:
            seg = seg.strip()
            if not seg:
                continue
            m = PROGRESS_RE.search(seg)
            if m:
                pct_str, processed_t, remaining_t, speed = m.groups()
                pct = int(pct_str)
                remaining_t = remaining_t.strip()
                if pct != last_pct:
                    last_pct = pct
                    bar_len = 25
                    filled = int(pct / 100 * bar_len)
                    bar = "█" * filled + "░" * (bar_len - filled)
                    print(f"  [{bar}] {pct:3d}%  已处理:{processed_t}  剩余:{remaining_t}  速度:{speed}")
            else:
                # 非进度行直接打印（过滤掉空行和纯分隔符）
                if seg and not re.match(r'^[|\s]+$', seg):
                    print(f"  ℹ️  {seg}")

    proc.wait()
    return proc.returncode

total_folders = len(FOLDERS)
for folder_idx, folder in enumerate(FOLDERS, 1):
    folder_path = Path(folder)
    folder_name = folder_path.name

    print(f"\n{'#'*65}")
    print(f"# [{folder_idx}/{total_folders}] {folder_name}")
    print(f"{'#'*65}")

    if not folder_path.exists():
        print(f"⚠️ 文件夹不存在，跳过")
        continue

    videos = []
    for f in sorted(folder_path.iterdir()):
        if f.suffix.lower() in VIDEO_EXTENSIONS and not f.stem.endswith(".MR"):
            if (f.parent / f"{f.stem}.MR.mp4").exists():
                print(f"⏭️ 已有输出，跳过: {f.name}")
            else:
                videos.append(f)

    if not videos:
        print("✅ 该文件夹无需处理")
        continue

    print(f"📋 待处理 {len(videos)} 个视频")

    # 步骤 1: 复制到本地磁盘（IO 更快）
    local_folder = LOCAL_WORK_DIR / folder_name
    local_folder.mkdir(parents=True, exist_ok=True)
    print(f"\n📥 复制到本地磁盘...")
    for v in videos:
        dst = local_folder / v.name
        if not dst.exists():
            print(f"  {v.name}  ({v.stat().st_size/1024/1024:.1f} MB)")
            shutil.copy2(v, dst)

    # 步骤 2: 逐个去马赛克
    print(f"\n🔧 开始去马赛克处理...")
    for vid_idx, orig_video in enumerate(videos, 1):
        local_input  = local_folder / orig_video.name
        local_output = local_folder / f"{orig_video.stem}.MR.mp4"

        print(f"\n{'─'*65}")
        print(f"  [{vid_idx}/{len(videos)}] {orig_video.name}")
        print(f"{'─'*65}")

        if local_output.exists():
            print("  ⏭️ 本地已有输出，跳过")
            continue

        start_time = time.time()
        returncode = process_video(local_input, local_output)
        elapsed = time.time() - start_time
        m_min, s = divmod(int(elapsed), 60)
        if returncode == 0:
            size_mb = local_output.stat().st_size / 1024 / 1024 if local_output.exists() else 0
            print(f"  ✅ 完成  耗时 {m_min}分{s:02d}秒  输出 {size_mb:.1f} MB")
        else:
            print(f"  ❌ 失败 (返回码 {returncode})  耗时 {m_min}分{s:02d}秒")

    # 步骤 3: 将输出移回 Drive
    print(f"\n📤 移回 Drive...")
    for orig_video in videos:
        local_output = local_folder / f"{orig_video.stem}.MR.mp4"
        drive_output = folder_path / f"{orig_video.stem}.MR.mp4"
        if local_output.exists():
            print(f"  {local_output.name}  ({local_output.stat().st_size/1024/1024:.1f} MB)")
            shutil.copy2(local_output, drive_output)
            local_output.unlink()
        else:
            print(f"  ⚠️ 未找到输出: {local_output.name}")

    # 步骤 4: 清理本地临时文件
    print(f"\n🗑️ 清理本地...")
    shutil.rmtree(local_folder)
    print(f"✅ [{folder_idx}/{total_folders}] {folder_name} 完成\n")

if LOCAL_WORK_DIR.exists() and not any(LOCAL_WORK_DIR.iterdir()):
    LOCAL_WORK_DIR.rmdir()

print(f"\n{'#'*65}")
print(f"🎉 全部完成！共 {total_folders} 个文件夹。")
print(f"{'#'*65}")